# 27 — M3 Gate

**Autoridad formal de cierre de M3.** Consume evidencia persistida por 20–26 y decide computablemente si Gold puede congelarse antes de Modeling.

In [1]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

{'gold_schema_version': 'm3-gold-v1',
 'target': 'y_addition_180d',
 'horizon_days': 180,
 'event_scope': ('ADICION EN EL VALOR',),
 'duckdb_threads': 2,
 'duckdb_memory_limit': '8GB',
 'm2_gate': '/Users/macm1pro/Desktop/projects/argos/argos-code/docs/decisions/M2_GATE.md'}

In [2]:
from argos.evaluation.m3_gate import evaluate_m3_gate, persist_m3_gate

report, decision = evaluate_m3_gate(
    reports_root=paths.reports,
    gold_dir=paths.gold,
    manifest_dir=paths.root / "artifacts" / "manifests",
    docs_root=paths.docs,
    schema_version=GOLD_SPEC.schema_version,
)
report

,check_id,category,blocking,passed,observed,requirement
0,notebook_20_blocking_checks,component,True,True,"{""blocking"": 13, ""failed"": [], ""passed"": 13}",notebook 20 has all blocking checks passed
1,notebook_21_blocking_checks,component,True,True,"{""blocking"": 7, ""failed"": [], ""passed"": 7}",notebook 21 has all blocking checks passed
2,notebook_22_blocking_checks,component,True,True,"{""blocking"": 3, ""failed"": [], ""passed"": 3}",notebook 22 has all blocking checks passed
3,notebook_23_blocking_checks,component,True,True,"{""blocking"": 4, ""failed"": [], ""passed"": 4}",notebook 23 has all blocking checks passed
4,notebook_24_blocking_checks,component,True,True,"{""blocking"": 3, ""failed"": [], ""passed"": 3}",notebook 24 has all blocking checks passed
5,notebook_25_blocking_checks,component,True,True,"{""blocking"": 3, ""failed"": [], ""passed"": 3}",notebook 25 has all blocking checks passed
6,notebook_26_blocking_checks,component,True,True,"{""blocking"": 8, ""failed"": [], ""passed"": 8}",notebook 26 has all blocking checks passed
7,gold_artifacts_complete,artifacts,True,True,"{""expected"": 7, ""missing"": []}",all canonical M3 Gold artifacts exist
8,gold_lineage_manifests_complete,lineage,True,True,"{""missing"": []}",all M3 component manifests are persisted
9,gold_schema_version_consistent,lineage,True,True,"{""errors"": [], ""expected"": ""m3-gold-v1""}",all M3 manifests declare the active Gold schem...


In [3]:
report_path, decision_path = persist_m3_gate(
    report=report,
    decision=decision,
    reports_root=paths.reports,
    decision_path=paths.docs / "decisions" / "M3_GATE.md",
)

print("M3 ✅ CLOSED" if decision.closed else "M3 ❌ OPEN")
print(f"Blocking checks: {decision.blocking_passed}/{decision.blocking_total}")
if decision.failed_checks:
    print("Failed:", ", ".join(decision.failed_checks))
print("Gate report:", report_path)
print("Decision artifact:", decision_path)

assert decision.closed, (
    "M3 remains OPEN. Resolve the failed checks before starting Modeling."
)

M3 ✅ CLOSED
Blocking checks: 13/13
Gate report: /Users/macm1pro/Desktop/projects/argos/argos-code/reports/gold/m3_gate_report.parquet
Decision artifact: /Users/macm1pro/Desktop/projects/argos/argos-code/docs/decisions/M3_GATE.md


## Después del cierre

Si el gate imprime `M3 ✅ CLOSED`, congelar Gold, el test temporal y los manifests. El siguiente milestone comienza con `30_baseline_heuristics.ipynb` y `31_logistic_regression_baseline.ipynb`.